<a href="https://colab.research.google.com/github/ryanatberkeley/aeesp-grand-challenges-nlp-pipeline/blob/main/aseep_cleanedcorpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAlex Bibliographic Corpus Builder

This notebook queries the OpenAlex API to collect publications for a researcher and build a structured corpus containing:

- titles
- abstracts
- keywords
- authors
- publication metadata

In [27]:
try:
    import pyalex
except ImportError:
    !pip install pyalex
    import pyalex

## Import libraries

In [28]:
import difflib
import io
import time

import pandas as pd
import pyalex
from pyalex import Authors, Institutions, Works

## Upload AEESP member CSV


In [29]:
df_authors = pd.read_csv("aeesp_members_short.csv")

df_authors.columns = df_authors.columns.str.strip().str.replace('\n', '')

print("\n--- Diagnostic Check ---")
print("Columns found in your CSV:", df_authors.columns.tolist())
print("------------------------\n")


--- Diagnostic Check ---
Columns found in your CSV: ['Last Name', 'First Name', 'Email', 'Job Institution', 'Department', 'Job Title', 'Country']
------------------------



In [30]:
df_authors.head()

,Last Name,First Name,Email,Job Institution,Department,Job Title,Country
0,Aich,Nirupam,nirupam.aich@unl.edu,University at Nebraska - Lincoln,Department of Civil and Environmental Engineering,Richard L. McNeel Associate Professor,United States
1,Alfredo,Katherine Ann,kalfredo@usf.edu,University of South Florida,Civil and Environmental Engineering,Assistant Professor,United States
2,Ashenafi,Eyosias,ashene2@rpi.edu,Rensselaer Polytechnic Institute,Civil and Environmental Engineering,Lecturer,United States
3,Boualavong,Jonathan,jboualav@buffalo.edu,University at Buffalo,"Civil, Structural, and Environmental Engineering",Assistant Professor,United States
4,Brazeau,Randi H,rbrazeau@msudenver.edu,Metropolitan State University of Denver,Earth & Atmospheric Sciences,Professor,United States


## Preview uploaded data

## Initialize storage for author dataframes

In [31]:
all_author_dfs = {}

## Configuration and helper functions

Edit the **configuration** block (constants and lookup tables) to adapt the corpus builder to another member list, venue rules, or metadata fields. Helpers reconstruct OpenAlex abstracts, resolve CSV column names, and apply shared filters.

In [32]:
# --- Pipeline configuration (adjust for your project) ---
# Placeholder text when OpenAlex has no inverted abstract (must match topic modeling notebook)
ABSTRACT_PLACEHOLDER = "No abstract available"

# Optional: set for OpenAlex "polite pool" (recommended for large author lists)
OPENALEX_MAILTO = None  # e.g. "you@university.edu"

# OpenAlex work `type` values to drop (e.g. preprints, theses)
NON_ARCHIVAL_WORK_TYPES = frozenset({"preprint", "dissertation"})

# Case-insensitive substrings matched against the venue/source `display_name`
NON_ARCHIVAL_VENUE_SUBSTRINGS = (
    "biorxiv",
    "medrxiv",
    "zenodo",
    "conference",
    "abstract",
    "thesis",
    "dissertation",
)

# Substrings matched against OpenAlex author `x_concepts` for disambiguation
AUTHOR_DISAMBIGUATION_KEYWORDS = (
    "environmental",
    "water",
    "engineering",
    "chemistry",
    "wastewater",
    "civil",
)

# Rows dropped after fetch (aligns with guided topic modeling needing title + abstract)
REQUIRE_NON_EMPTY_TITLE = True
REQUIRE_ABSTRACT = True

EXCLUDED_AUTHORS = {
    n.lower().strip()
    for n in [
        "Xioahong Chen",
        "Peng Wang",
        "Kaye Reed",
        "Tae-Kyoung Kim",
        "Chang-Yu Wu",
        "M. Kabir Hassan",
        "Ying Wang",
        "Xun Yang",
        "Stephen Maisto",
        "Zhong Lin Wang",
        "Qingqing Sun",
        "Yutong Liu",
        "Sanu Lama",
        "Richard Dick",
        "Katherine McMahon",
        "Thomas Young",
    ]
}


def reconstruct_abstract(inverted_index, placeholder=ABSTRACT_PLACEHOLDER):
    if not inverted_index:
        return placeholder

    max_index = max(max(indices) for indices in inverted_index.values())
    abstract_list = [""] * (max_index + 1)

    for word, indices in inverted_index.items():
        for index in indices:
            abstract_list[index] = word

    return " ".join(abstract_list)


def string_similarity(s1, s2):
    return difflib.SequenceMatcher(None, str(s1).lower(), str(s2).lower()).ratio()


def clean_alternative_names(primary_name, alternatives, threshold=0.5):
    if not alternatives:
        return []

    cleaned = []
    for alt in alternatives:
        if string_similarity(primary_name, alt) >= threshold:
            cleaned.append(alt)

    return cleaned


def normalize_whitespace(value):
    return " ".join(str(value).strip().split())


def clean_person_name(value):
    # Remove punctuation noise while preserving initials and spacing.
    cleaned = normalize_whitespace(value)
    cleaned = cleaned.strip(". ,;:-")
    return cleaned


def normalize_institution_name(value):
    value = normalize_whitespace(value)
    value = value.replace("–", "-").replace("—", "-")
    value = " ".join(value.split())
    return value


def build_institution_search_candidates(raw_value):
    base = normalize_institution_name(raw_value)

    split_tokens = ["|", "/", "(", ";"]

    candidates = []

    if "|" in base:
        candidates.append(base.split("|")[0].strip())
    else:
        candidates.append(base)

    for token in split_tokens:
        if token in base:
            candidates.append(base.split(token)[0].strip())

    if "," in base:
        candidates.append(base.split(",")[0].strip())

    if base.isupper():
        candidates.append(base.title())

    deduped = []
    for candidate in candidates:
        candidate = normalize_whitespace(candidate)
        if candidate and candidate not in deduped:
            deduped.append(candidate)

    return deduped


def select_best_institution(inst_search, target_name):
    target_name = normalize_institution_name(target_name).lower()

    best = None
    best_score = float("-inf")

    for inst in inst_search:
        display_name = normalize_institution_name(inst.get("display_name", "")).lower()

        score = string_similarity(target_name, display_name)

        if target_name in display_name or display_name in target_name:
            score += 0.25

        target_words = set(target_name.split())
        display_words = set(display_name.split())
        overlap = len(target_words & display_words) / max(len(target_words), 1)
        score += 0.25 * overlap

        works_count = inst.get("works_count", 0) or 0
        score += min(works_count / 1_000_000, 0.10)

        if score > best_score:
            best_score = score
            best = inst

    return best


def resolve_institution_record(raw_value):
    candidates = build_institution_search_candidates(raw_value)

    all_results = []

    for candidate in candidates:
        inst_search = Institutions().search(candidate).get()
        for inst in inst_search:
            all_results.append((candidate, inst))

    if not all_results:
        return candidates[0] if candidates else normalize_whitespace(raw_value), None

    best_query, _ = max(
        all_results,
        key=lambda pair: string_similarity(
            normalize_institution_name(pair[0]).lower(),
            normalize_institution_name(pair[1].get("display_name", "")).lower()
        )
    )

    best_inst = select_best_institution(
        [inst for _, inst in all_results],
        raw_value
    )

    return normalize_institution_name(raw_value), best_inst


def build_author_query_candidates(first_name, last_name):
    first_clean = clean_person_name(first_name)
    last_clean = clean_person_name(last_name)

    first_parts = [p for p in first_clean.split(" ") if p]
    last_parts = [p for p in last_clean.split(" ") if p]

    first_primary = first_parts[0] if first_parts else first_clean
    last_primary = last_parts[-1] if last_parts else last_clean

    candidates = [
        f"{first_clean} {last_clean}".strip(),
        f"{first_primary} {last_primary}".strip(),
    ]

    if first_primary:
        candidates.append(f"{first_primary[0]} {last_primary}".strip())

    deduped = []
    for candidate in candidates:
        c = normalize_whitespace(candidate)
        if c and c not in deduped:
            deduped.append(c)
    return deduped


def select_best_author(auth_search, target_first, target_last):
    target_first = clean_person_name(target_first)
    target_last = clean_person_name(target_last)

    best = None
    best_score = float("-inf")

    for potential_author in auth_search:
        display_name = potential_author.get("display_name", "")
        concepts = [c["display_name"].lower() for c in potential_author.get("x_concepts", [])]

        name_score = string_similarity(f"{target_first} {target_last}", display_name)
        keyword_bonus = 0.0
        if any(keyword in " ".join(concepts) for keyword in AUTHOR_DISAMBIGUATION_KEYWORDS):
            keyword_bonus = 0.35

        score = name_score + keyword_bonus
        if score > best_score:
            best_score = score
            best = potential_author

    return best


def _resolve_column(df, candidates):
    for name in candidates:
        if name in df.columns:
            return name
    return None


def host_organization_from_source(source):
    if not source:
        return None
    name = source.get("host_organization_name")
    if name:
        return name
    org = source.get("host_organization") or {}
    if isinstance(org, dict):
        return org.get("display_name")
    return None


def format_coauthors_for_work(work, focal_author_id):
    """Names and affiliations of co-authors (all authorships except the focal author OpenAlex id)."""
    authorships = work.get("authorships") or []
    names = []
    aff_strings = []
    for authorship in authorships:
        auth = authorship.get("author") or {}
        aid = auth.get("id")
        if focal_author_id and aid == focal_author_id:
            continue
        name = auth.get("display_name") or authorship.get("raw_author_name") or ""
        insts = authorship.get("institutions") or []
        inst_names = ", ".join(
            sorted({i.get("display_name") for i in insts if i and i.get("display_name")})
        )
        name = name.strip()
        if name:
            names.append(name)
            aff_strings.append(inst_names if inst_names else "")
    return "; ".join(names), "; ".join(aff_strings)


def should_exclude_work(work):
    if work.get("type") in NON_ARCHIVAL_WORK_TYPES:
        return True
    primary_loc = work.get("primary_location") or {}
    source = primary_loc.get("source") or {}
    venue_display_name = source.get("display_name")
    pub_lower = venue_display_name.lower() if venue_display_name else ""
    return any(s in pub_lower for s in NON_ARCHIVAL_VENUE_SUBSTRINGS)


def apply_post_fetch_filters(df):
    """Drop rows missing title or real abstract when REQUIRE_* flags are set."""
    if df.empty:
        return df
    out = df
    if REQUIRE_NON_EMPTY_TITLE:
        title_str = out["title"].astype(str).str.strip()
        out = out[out["title"].notna() & (title_str != "")]
    if REQUIRE_ABSTRACT:
        out = out[out["abstract"].notna() & (out["abstract"] != ABSTRACT_PLACEHOLDER)]
        abs_str = out["abstract"].astype(str).str.strip()
        out = out[abs_str != ""]
    return out

## Retrieve works for each AEESP member

For each row in the CSV:

1. Search OpenAlex for the institution
2. Search OpenAlex for the author within that institution
3. Retrieve all works associated with the author
4. Extract metadata (venue host org via `host_organization_name`; co-authors and their institutions from `authorships`)
5. Reconstruct abstracts
6. Keep filtered alternate author names
7. Remove duplicate works
8. Store results as a dataframe

In [33]:
if OPENALEX_MAILTO:
    pyalex.config.email = OPENALEX_MAILTO

first_col = _resolve_column(df_authors, ("First Name", "first_name", "FirstName"))
last_col = _resolve_column(df_authors, ("Last Name", "last_name", "LastName"))
inst_col = _resolve_column(
    df_authors,
    ("Institution", "Job Institution", "Job institution", "institution"),
)
if not first_col or not last_col or not inst_col:
    raise ValueError(
        "Could not resolve CSV columns for first name, last name, and institution. "
        f"Found columns: {list(df_authors.columns)}"
    )

for index, row in df_authors.iterrows():
    first_name = clean_person_name(row[first_col])
    last_name = clean_person_name(row[last_col])
    raw_institution = str(row[inst_col]).strip()

    print(f"Processing: {first_name} {last_name} at {normalize_whitespace(raw_institution)}...")

    try:
        # 1) Institution resolution with multiple normalized candidates.
        matched_institution_query, inst_record = resolve_institution_record(raw_institution)
        if not inst_record:
            print(
                f"  -> Institution not found from candidates: "
                f"{build_institution_search_candidates(raw_institution)}. Skipping."
            )
            continue

        inst_id = inst_record["id"]

        # 2) Author search with multiple query variants.
        author_candidates = []
        author_query_variants = build_author_query_candidates(first_name, last_name)
        for query in author_query_variants:
            results = Authors().search(query).filter(
                affiliations={"institution": {"id": inst_id}}
            ).get()
            if results:
                author_candidates.extend(results)

        # 3) Fallback: loose search by last name at institution.
        if not author_candidates:
            short_last = clean_person_name(last_name).split()[-1] if last_name else ""
            if short_last:
                print(f"  -> Variant search failed. Retrying with last-name fallback '{short_last}'...")
                author_candidates = Authors().search(short_last).filter(
                    affiliations={"institution": {"id": inst_id}}
                ).get()

        if not author_candidates:
            print(
                f"  -> Author '{first_name} {last_name}' not found at '{matched_institution_query}'. Skipping."
            )
            continue

        # 4) Pick best candidate using name similarity + domain-keyword bonus.
        auth = select_best_author(author_candidates, first_name, last_name)
        if not auth:
            print(f"  -> Warning: No disambiguation winner for {first_name} {last_name}. Using first result.")
            auth = author_candidates[0]

        author_id = auth['id']
        matched_author_name = auth.get('display_name')
        matched_author_orcid = auth.get('orcid')
        author_works_count = auth.get('works_count', 0)

        # Filter out authors based on works_count (0)
        if author_works_count == 0:
            print(f"  -> Skipping '{matched_author_name}' due to 0 publications.")
            continue

        if matched_author_name and matched_author_name.lower().strip() in EXCLUDED_AUTHORS:
            print(f"  -> Skipping '{matched_author_name}' (manual exclusion list).")
            continue

        display_name_alternatives = clean_alternative_names(
            matched_author_name,
            auth.get('display_name_alternatives', [])
        )

        # 5) Retrieve works
        works_iterator = Works().filter(
            authorships={"author": {"id": author_id}}
        ).paginate(per_page=100)

        works_data = []

        for page in works_iterator:
            for work in page:
                if should_exclude_work(work):
                    continue

                primary_loc = work.get('primary_location') or {}
                source = primary_loc.get('source') or {}
                host_organization_name = host_organization_from_source(source)

                co_authors_str, co_author_institutions_str = format_coauthors_for_work(
                    work, author_id
                )

                biblio = work.get('biblio') or {}
                volume = biblio.get('volume')
                issue = biblio.get('issue')
                first_page = biblio.get('first_page')
                last_page = biblio.get('last_page')

                keywords_list = work.get('keywords') or []
                keywords = ", ".join([k.get('display_name', '') for k in keywords_list])

                abstract_inverted_index = work.get('abstract_inverted_index')
                has_abstract = abstract_inverted_index is not None
                abstract = reconstruct_abstract(abstract_inverted_index)

                works_data.append({
                    'id': work.get('id'),
                    'doi': work.get('doi'),
                    'title': work.get('title'),
                    'work_type': work.get('type'),
                    'publication_year': work.get('publication_year'),
                    'host_organization_name': host_organization_name,
                    'co_authors': co_authors_str,
                    'co_author_institutions': co_author_institutions_str,
                    'volume': volume,
                    'issue': issue,
                    'first_page': first_page,
                    'last_page': last_page,
                    'keywords': keywords,
                    'open_access': work.get('open_access', {}).get('is_oa'),
                    'abstract': abstract,
                    'has_abstract': has_abstract,
                    'matched_author_name': matched_author_name,
                    'matched_author_orcid': matched_author_orcid,
                    'display_name_alternatives': "; ".join(display_name_alternatives)
                })

        df = pd.DataFrame(works_data)

        if not df.empty:
            df = apply_post_fetch_filters(df)

            df = df.sort_values(
              by=['has_abstract', 'publication_year'],
              ascending=[False, False]
            )

            df = df.drop_duplicates(subset=['id'], keep='first')

            if 'doi' in df.columns:
                df = df.drop_duplicates(subset=['doi'], keep='first')

            df = df.drop_duplicates(subset=['title'], keep='first')

        safe_last = last_name.lower().replace(' ', '').replace('-', '')
        safe_first = first_name.lower().replace(' ', '').replace('-', '')

        df_variable_name = f"works_df_{safe_last}_{safe_first}"

        globals()[df_variable_name] = df
        all_author_dfs[df_variable_name] = df

        print(f"  -> Success! Created `{df_variable_name}` with {len(df)} unique publications.")

    except Exception as e:
        print(f"  -> Error processing {first_name} {last_name}: {e}")

    time.sleep(1)

print("\nFinished processing all authors!")

Processing: Nirupam Aich at University at Nebraska - Lincoln...
  -> Institution not found from candidates: ['University at Nebraska - Lincoln']. Skipping.
Processing: Katherine Ann Alfredo at University of South Florida...
  -> Success! Created `works_df_alfredo_katherineann` with 21 unique publications.
Processing: Eyosias Ashenafi at Rensselaer Polytechnic Institute...
  -> Success! Created `works_df_ashenafi_eyosias` with 4 unique publications.
Processing: Jonathan Boualavong at University at Buffalo...
  -> Success! Created `works_df_boualavong_jonathan` with 7 unique publications.
Processing: Randi H Brazeau at Metropolitan State University of Denver...
  -> Success! Created `works_df_brazeau_randih` with 8 unique publications.
Processing: Elizabeth Butler at University of Oklahoma...
  -> Success! Created `works_df_butler_elizabeth` with 40 unique publications.
Processing: Jose M Cerrato at University of New Mexico...
  -> Success! Created `works_df_cerrato_josem` with 60 unique

## Export all author data

Each dataframe beginning with `works_df_` will be saved as a CSV file and packaged into a zip archive.

In [34]:
import os
import zipfile

print("Gathering and zipping all author data...")

zip_filename = "all_authors_works.zip"

with zipfile.ZipFile(zip_filename, "w") as zipf:
    for var_name in list(globals().keys()):
        if var_name.startswith("works_df_"):
            df = globals()[var_name]

            csv_filename = f"{var_name}.csv"
            df.to_csv(csv_filename, index=False)

            zipf.write(csv_filename)
            os.remove(csv_filename)

print(f"Successfully created {zip_filename}!")

ModuleNotFoundError: No module named 'google.colab'